[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/04_grounded_data_augmentation.ipynb)

# Step 4 — Retrieval-Augmented Synthetic Data

Generate **grounded** Q&A from TRAIN paragraphs to build the SFT corpus (target: 500–1,000 samples).

## Learning objectives
- Retrieve relevant passages from policy documents
- Generate faithful Q&A pairs with a teacher LLM
- Verify grounding with lexical overlap or embedding similarity

Step 2 data teaches how to prompt; Step 4 data is what you train on
Grounded generation + overlap check reduces hallucinated training labels
In a real deployment you'd have more documents and actually hit 500–1000; the bootcamp simulates the pipeline at small scale


In [1]:
from pathlib import Path
import os

from aieng.syn_data.text import (
    DEFAULT_SYNTHETIC_TARGET_SIZE,
    PARAGRAPHS_PATH,
    SYNTHETIC_TRAIN_PATH,
    Paragraph,
    ParagraphSplit,
    QASample,
    apply_heuristic_filters,
    create_teacher_client,
    effective_synthetic_target,
    generate_grounded_training_corpus,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
)
from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table

# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


In [2]:
# TODO: remove this before merging into main
%load_ext autoreload
%autoreload 2

## 1. Load TRAIN paragraphs (exclude test holdout)

In [3]:
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
train_paragraphs = [p for p in all_paragraphs if p.split == ParagraphSplit.TRAIN]
console.print(f"Train paragraphs: {len(train_paragraphs)}")

Train paragraphs: 36

## 2. Generate grounded Q&A from retrieved passages

So notebook 02 → 03 walks you through the canonical "generate everything, then filter with heuristics + judge" pipeline as a learning exercise. Notebook 04 is the alternative: bake grounding constraints into generation itself so you get a usable training corpus directly, without needing the downstream judge filter to throw most of it away. The training file you already saved came from the 02→03 path; notebook 04 produces a different corpus using the grounded approach.

Grounded-RAG in rag.py adds two extra things on top:

1. A prompt that explicitly says "The answer must be fully supported by the passage" (grounded_qa_prompt), forcing the model into an extractive/faithful mode rather than free-form generation.

2. A post-hoc lexical overlap check (grounding_overlap_score) that rejects samples whose answer tokens don't sufficiently appear in the passage.


In [4]:
teacher = create_teacher_client()
target_size = effective_synthetic_target(
    train_paragraphs,
    requested=DEFAULT_SYNTHETIC_TARGET_SIZE,
)
print(f"Grounded generation target: {target_size}")

grounded_candidates = generate_grounded_training_corpus(
    teacher,
    train_paragraphs,
    target_size=target_size,
    min_overlap=0.15,
)
print(f"Generated {len(grounded_candidates)} grounded candidates")
grounded_candidates[0].metadata

Grounded generation target: 100


2026-06-25 22:51:52,117 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
  "instruction": "Based on the provided text, identify the conditions under which a user will be considered in default, and explain what rights the Credit Union has once a default occurs.",
  "question": "According to the policy, under what circumstances will you be in default, and what actions can the Credit Union take immediately upon default?",
  "gold_answer": "You will be in default if you: fail to make any minimum or required payment by the due date; break any promise made under the Agreement; die, file for bankruptcy, or become insolvent; make false or misleading statements in a credit application or update; or if something happens that the Credit Union believes may substantially reduce your ability to repay. Once you are in default, the Credit Union has the right to demand immediate payment of your full account balance without giving you notice."
}
*********** End of JS

Generated 100 grounded candidates


{'generation_strategy': 'grounded_rag', 'grounding_overlap': 0.95}

## 3. Filter and save final SFT corpus

In [5]:
final_samples, rejected = apply_heuristic_filters(grounded_candidates)
print(f"Final SFT corpus size: {len(final_samples)} (rejected {len(rejected)})")

save_typed_jsonl(
    SYNTHETIC_TRAIN_PATH,
    final_samples,
    to_dict=QASample.to_dict,
)
SYNTHETIC_TRAIN_PATH

Final SFT corpus size: 36 (rejected 64)


PosixPath('implementations/qa_text_generation/data/synthetic/synthetic_train.jsonl')